In [ ]:
import nltk
from nltk.corpus import state_union
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import gensim
from gensim import corpora
import matplotlib.pyplot as plt
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
import re

In [ ]:
import glob

In [ ]:
# Get a list of all text files in the data/dickens/ directory
file_paths = glob.glob('data/dickens/*.txt')
addresses = [name.split("/dickens/")[1] for name in file_paths]
# Read each file as a string and create an array of strings
file_contents = []
rawdocuments = file_contents
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as file:
        file_contents.append(file.read())

In [ ]:
documents = [doc.lower() for doc in rawdocuments]

In [ ]:
# Assuming 'documents' is your list of documents
documents = [re.sub(r'[^\w\s]', '', doc) for doc in documents]

In [ ]:
stop_words = stopwords.words('english')

In [ ]:
#filter stops and punctuation
tokenizedCorpus = []
for doc in documents:
  tokens = word_tokenize(doc)
  withOutStops = [word for word in doc.split() if word not in stop_words]
  tokenizedCorpus.append(withOutStops)

In [ ]:
from collections import Counter
words = [word for doc in tokenizedCorpus for word in doc]
word_counts = Counter(words)
common_words = [word for word, freq in word_counts.most_common(1000)]
least_common_words = [word for word, freq in word_counts.items() if freq <= 10]
tokenizedCorpus = [[word for word in doc if word not in common_words] for doc in tokenizedCorpus]
tokenizedCorpus = [[word for word in doc if word not in least_common_words] for doc in tokenizedCorpus]

In [ ]:
common_words

In [ ]:
#build dictionary of corpus for later use to decode topics from word vectors
dictionary = corpora.Dictionary(tokenizedCorpus)

In [ ]:
#convert corpus to a bag of words model
doc_term_matrix = [dictionary.doc2bow(doc) for doc in tokenizedCorpus]

In [ ]:
#instantiate the lda algo
lda = gensim.models.ldamodel.LdaModel
#run the topic model! 20 passes is standard, you may wish to try more as needed
ldamodel = (lda(doc_term_matrix, num_topics=15, id2word=dictionary, passes=100))
#export topics to a list of tuples
topics = ldamodel.print_topics(num_topics=15, num_words=15)
#print topics in readable format
for top in topics:
    print(top)

In [ ]:
vis_data = gensimvis.prepare(ldamodel, doc_term_matrix, dictionary, sort_topics=False)

In [ ]:
pyLDAvis.display(vis_data)

In [ ]:

# Get the documents for a given topic
#document_topics = [max(ldamodel.get_document_topics(bow), key=lambda x: x[1]) for bow in doc_term_matrix]
#documents_for_given_topic = [addresses[i] for i, topics in enumerate(document_topics) if topics[0] == given_topic]
given_topic = 13  # Change this to your given topic in pyLDAvis
documents_with_given_topic = [i for i, bow in enumerate(doc_term_matrix) if any(topic == (given_topic - 1) for topic, prob in ldamodel.get_document_topics(bow))]



In [ ]:
for i in documents_with_given_topic:
    print(addresses[i])

In [ ]:
resultList = []
for i, docs in enumerate(tokenizedCorpus):
  doc_topics, word_topics, phi_values = ldamodel.get_document_topics(doc_term_matrix[i], per_word_topics=True)
  resultList.append(sorted(doc_topics,key=lambda x: x[1], reverse=True))


In [ ]:
#print out each topic with its top 15 words, then underneat that print each document for which this topic is the number one dominant topic
for i, topic in enumerate(resultList):
  print(f"Document {addresses[i]} is about topic {topic[0][0]}")
  print(f"Topic {topic[0][0]}: {ldamodel.print_topic(topic[0][0])}")
  print()
  print()

In [ ]:
import matplotlib.pyplot as plt

# Get the most dominant topic for each document
document_topics = [max(ldamodel.get_document_topics(bow), key=lambda x: x[1])[0] for bow in doc_term_matrix]

# Extract the year from each address
years = [int(address.split('-')[0]) for address in addresses]

# Ensure that the lengths of the years and topics lists are the same
assert len(years) == len(document_topics), "The lengths of the years and topics lists do not match."

# Plot the most dominant topic in a bar graph over time
plt.figure(figsize=(12, 6))
plt.plot(years, document_topics)
plt.xlabel('Year')
plt.ylabel('Topic')
plt.title('Most Dominant Topic Over Time')
plt.show()


In [ ]:
for index, result in enumerate(resultList):
  print("document" + addresses[index])
  print(result)

In [ ]:
print(state_union.raw("1945-Truman.txt"))

In [ ]:
addresses